### 1. Introduction

This notebook develops and evaluates baseline classification models for predicting session-level conversion using the processed RetailRocket dataset. The baseline models provide reference points against which the subsequent Bayesian Network model can be evaluated.

Two baseline classifiers are considered: Logistic Regression, representing a relatively simple and interpretable linear classification approach, and Random Forest, representing a more flexible non-linear approach capable of modelling interactions between predictors.

Given the substantial class imbalance identified during preprocessing, model performance will not be assessed using accuracy alone. Evaluation will consider precision, recall, F1-score, ROC-AUC and precision-recall AUC, with particular attention given to performance on the minority conversion class.

### 2. Import Libraries

The libraries required for data manipulation, preprocessing, model development and evaluation are imported below.

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss
)

### 3. Load Processed Dataset

The processed session-level dataset produced during preprocessing and session construction is loaded for baseline modelling. This dataset retains the engineered session-level variables prior to discretisation, allowing the baseline classifiers to make use of the original numerical representations where appropriate.

In [3]:
session_features = pd.read_csv(
    "../data/processed/session_features.csv"
)

print(f"Dataset shape: {session_features.shape}")

session_features.head()

Dataset shape: (1761675, 17)


,session_id,visitorid,session_start,session_end,event_count,unique_items,session_duration_minutes,addtocart_count,transaction_count,view_count,converted,has_addtocart,session_hour,day_of_week,is_weekend,event_count_group,unique_items_group
0,0_1,0,2015-09-11 20:49:49.439,2015-09-11 20:55:17.175,3,3,5.462267,0,0,3,0,0,20,Friday,0,3,3
1,1000000_1,1000000,2015-06-05 18:16:10.629,2015-06-05 18:16:10.629,1,1,0.000000,0,0,1,0,0,18,Friday,0,1,1
2,1000001_1,1000001,2015-07-07 18:12:14.953,2015-07-07 18:12:14.953,1,1,0.000000,0,0,1,0,0,18,Tuesday,0,1,1
3,1000001_2,1000001,2015-07-24 20:18:15.303,2015-07-24 20:35:57.029,3,3,17.695433,0,0,3,0,0,20,Friday,0,3,3
4,1000001_3,1000001,2015-07-29 20:38:29.170,2015-07-29 20:38:29.170,1,1,0.000000,0,0,1,0,0,20,Wednesday,0,1,1


In [4]:
session_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1761675 entries, 0 to 1761674
Data columns (total 17 columns):
 #   Column                    Dtype  
---  ------                    -----  
 0   session_id                object 
 1   visitorid                 int64  
 2   session_start             object 
 3   session_end               object 
 4   event_count               int64  
 5   unique_items              int64  
 6   session_duration_minutes  float64
 7   addtocart_count           int64  
 8   transaction_count         int64  
 9   view_count                int64  
 10  converted                 int64  
 11  has_addtocart             int64  
 12  session_hour              int64  
 13  day_of_week               object 
 14  is_weekend                int64  
 15  event_count_group         object 
 16  unique_items_group        object 
dtypes: float64(1), int64(10), object(6)
memory usage: 228.5+ MB


In [5]:
session_features.columns.tolist()

['session_id',
 'visitorid',
 'session_start',
 'session_end',
 'event_count',
 'unique_items',
 'session_duration_minutes',
 'addtocart_count',
 'transaction_count',
 'view_count',
 'converted',
 'has_addtocart',
 'session_hour',
 'day_of_week',
 'is_weekend',
 'event_count_group',
 'unique_items_group']

### 4. Prepare Modelling Dataset

The session-level dataset contains variables that are not appropriate as predictors, including identifiers, raw timestamps and variables that directly reveal the conversion outcome. Predictor selection was therefore performed before model development to reduce data leakage and avoid redundant representations.

The transaction count was excluded because the conversion target is directly derived from the presence of a transaction and would therefore constitute target leakage. Session and visitor identifiers were also excluded because they identify individual observations or users rather than representing generalisable behavioural characteristics.

The baseline models use session engagement, cart behaviour and temporal characteristics as predictors of conversion.

#### 4.1 Predictor Selection

In [6]:
predictor_variables = [
    "event_count",
    "unique_items",
    "session_duration_minutes",
    "has_addtocart",
    "session_hour",
    "day_of_week"
]

target_variable = "converted"

X = session_features[predictor_variables].copy()
y = session_features[target_variable].copy()

print(f"Predictor dataset shape: {X.shape}")
print(f"Target shape: {y.shape}")

X.head()

Predictor dataset shape: (1761675, 6)
Target shape: (1761675,)


,event_count,unique_items,session_duration_minutes,has_addtocart,session_hour,day_of_week
0,3,3,5.462267,0,20,Friday
1,1,1,0.000000,0,18,Friday
2,1,1,0.000000,0,18,Tuesday
3,3,3,17.695433,0,20,Friday
4,1,1,0.000000,0,20,Wednesday


#### 4.2 Target Distribution

The distribution of the binary conversion target was reviewed prior to splitting the data to confirm the degree of class imbalance identified during preprocessing.

In [7]:
target_counts = y.value_counts().sort_index()
target_percentages = y.value_counts(normalize=True).sort_index() * 100

print("Target counts:")
print(target_counts)

print("\nTarget percentages:")
print(target_percentages.round(2))

Target counts:
converted
0    1747378
1      14297
Name: count, dtype: int64

Target percentages:
converted
0    99.19
1     0.81
Name: proportion, dtype: float64


#### 4.3 Train-Test Split

The dataset was divided into training and test sets so that model performance could be evaluated on observations not used during model fitting. A stratified split was used to preserve the highly imbalanced conversion distribution within both subsets.

A fixed random state was specified to ensure that the split is reproducible.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training observations: {len(X_train):,}")
print(f"Test observations: {len(X_test):,}")

print("\nTraining conversion rate:")
print((y_train.mean() * 100).round(2), "%")

print("\nTest conversion rate:")
print((y_test.mean() * 100).round(2), "%")

Training observations: 1,409,340
Test observations: 352,335

Training conversion rate:
0.81 %

Test conversion rate:
0.81 %


#### 4.4 Preprocessing Pipeline

A preprocessing pipeline was constructed to prepare the predictor variables for the baseline models. Numerical variables were standardised to place variables measured on different scales onto a comparable scale. The binary add-to-cart indicator was retained without transformation.

Day of week was treated as a categorical rather than ordinal variable because the numerical ordering of weekdays would not represent a meaningful linear relationship. It was therefore transformed using one-hot encoding, with one category dropped to avoid redundant dummy variables.

The preprocessing transformations were fitted using the training data only to prevent information from the test set influencing model development.

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_features = [
    "event_count",
    "unique_items",
    "session_duration_minutes",
    "session_hour"
]

binary_features = [
    "has_addtocart"
]

categorical_features = [
    "day_of_week"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("binary", "passthrough", binary_features),
        ("categorical", OneHotEncoder(
            drop="first",
            handle_unknown="ignore"
        ), categorical_features)
    ]
)

preprocessor

ColumnTransformer(transformers=[('numeric', StandardScaler(),
                                 ['event_count', 'unique_items',
                                  'session_duration_minutes', 'session_hour']),
                                ('binary', 'passthrough', ['has_addtocart']),
                                ('categorical',
                                 OneHotEncoder(drop='first',
                                               handle_unknown='ignore'),
                                 ['day_of_week'])])

### 5. Baseline Model: Logistic Regression

#### 5.1 Model Construction

Logistic regression was selected as an interpretable baseline classifier for predicting session-level conversion. Establishing a baseline provides a reference against which the predictive performance of the Bayesian Network can subsequently be compared.

The preprocessing pipeline defined previously was combined with the logistic regression classifier to ensure that all transformations were learned exclusively from the training data. The model was initially fitted without class weighting to provide a standard baseline before considering approaches to address the substantial class imbalance identified during exploratory analysis.

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ]
)

logistic_model

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric', StandardScaler(),
                                                  ['event_count',
                                                   'unique_items',
                                                   'session_duration_minutes',
                                                   'session_hour']),
                                                 ('binary', 'passthrough',
                                                  ['has_addtocart']),
                                                 ('categorical',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['day_of_week'])])),
                ('classifier',
                 LogisticRegression(max_iter=1000, random_state=42))])

In [11]:
# Fit baseline logistic regression model
logistic_model.fit(X_train, y_train)

print("Logistic regression model fitted successfully.")

Logistic regression model fitted successfully.


#### 5.2 Baseline Model Performance

The fitted logistic regression model was evaluated on the held-out test set. Given the substantial class imbalance, accuracy alone would provide a misleading assessment of predictive performance because a model predicting all sessions as non-converting would achieve very high accuracy.

Performance was therefore assessed using precision, recall, F1-score, ROC-AUC and average precision (PR-AUC), alongside the confusion matrix. These metrics provide a more informative assessment of the model's ability to identify the relatively rare converting sessions.

In [12]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

# Generate predictions
y_pred = logistic_model.predict(X_test)
y_prob = logistic_model.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"PR-AUC:    {pr_auc:.4f}")

Accuracy:  0.9927
Precision: 0.6391
Recall:    0.2193
F1-score:  0.3266
ROC-AUC:   0.9502
PR-AUC:    0.4776


In [13]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

print("Confusion matrix:")
print(cm)

print("\nClassification report:")
print(
    classification_report(
        y_test,
        y_pred,
        digits=4,
        zero_division=0
    )
)

Confusion matrix:
[[349122    354]
 [  2232    627]]

Classification report:
              precision    recall  f1-score   support

           0     0.9936    0.9990    0.9963    349476
           1     0.6391    0.2193    0.3266      2859

    accuracy                         0.9927    352335
   macro avg     0.8164    0.6091    0.6614    352335
weighted avg     0.9908    0.9927    0.9909    352335



The baseline logistic regression model achieved an accuracy of 99.27%. However, this result should be interpreted cautiously given that only approximately 0.81% of sessions converted.

The model achieved a ROC-AUC of 0.9502, indicating strong ability to rank converting sessions above non-converting sessions across classification thresholds. The PR-AUC was 0.4776, substantially higher than the underlying conversion prevalence, further indicating useful discrimination for the minority class.

At the default classification threshold of 0.5, precision for converted sessions was 63.91%, while recall was considerably lower at 21.93%. Of the 2,859 converting sessions in the test set, 627 were correctly identified and 2,232 were incorrectly classified as non-converting. This resulted in an F1-score of 0.3266.

These results suggest that although the baseline model provides strong overall discrimination, the default decision threshold results in relatively conservative conversion predictions and fails to identify a large proportion of actual conversions. The effect of class imbalance therefore requires further consideration.

#### 5.3 Class-Weighted Logistic Regression

Given the low recall observed for the minority conversion class, a second logistic regression model was fitted using balanced class weights. This approach assigns greater importance to observations from the minority class during model fitting without altering the underlying training dataset.

The class-weighted model was evaluated using the same held-out test set and performance metrics as the unweighted baseline, allowing the effect of accounting for class imbalance to be assessed directly.

In [14]:
weighted_logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=42,
            class_weight="balanced"
        ))
    ]
)

weighted_logistic_model.fit(X_train, y_train)

print("Class-weighted logistic regression model fitted successfully.")

Class-weighted logistic regression model fitted successfully.


In [15]:
# Generate predictions
y_pred_weighted = weighted_logistic_model.predict(X_test)
y_prob_weighted = weighted_logistic_model.predict_proba(X_test)[:, 1]

# Calculate metrics
weighted_accuracy = accuracy_score(y_test, y_pred_weighted)
weighted_precision = precision_score(
    y_test, y_pred_weighted, zero_division=0
)
weighted_recall = recall_score(
    y_test, y_pred_weighted, zero_division=0
)
weighted_f1 = f1_score(
    y_test, y_pred_weighted, zero_division=0
)
weighted_roc_auc = roc_auc_score(
    y_test, y_prob_weighted
)
weighted_pr_auc = average_precision_score(
    y_test, y_prob_weighted
)

print(f"Accuracy:  {weighted_accuracy:.4f}")
print(f"Precision: {weighted_precision:.4f}")
print(f"Recall:    {weighted_recall:.4f}")
print(f"F1-score:  {weighted_f1:.4f}")
print(f"ROC-AUC:   {weighted_roc_auc:.4f}")
print(f"PR-AUC:    {weighted_pr_auc:.4f}")

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred_weighted))

Accuracy:  0.9644
Precision: 0.1694
Recall:    0.8671
F1-score:  0.2834
ROC-AUC:   0.9567
PR-AUC:    0.3941

Confusion matrix:
[[337322  12154]
 [   380   2479]]


Introducing balanced class weights substantially changed the classification behaviour of the logistic regression model. Recall for converting sessions increased from 21.93% to 86.71%, with the weighted model correctly identifying 2,479 of the 2,859 conversions in the test set. The number of false negatives consequently decreased from 2,232 to 380.

This improvement in sensitivity was accompanied by a substantial reduction in precision, from 63.91% to 16.94%, as the number of non-converting sessions incorrectly classified as conversions increased considerably. Overall accuracy also decreased from 99.27% to 96.44%.

ROC-AUC increased slightly from 0.9502 to 0.9567, indicating that the weighted model retained strong discrimination between converting and non-converting sessions. However, PR-AUC decreased from 0.4776 to 0.3941 and the F1-score decreased from 0.3266 to 0.2834. Therefore, class weighting improved the model's ability to identify actual conversions but did not improve performance across all evaluation measures.

The comparison demonstrates the trade-off between precision and recall when modelling a highly imbalanced outcome. Both logistic regression specifications are therefore retained as useful reference points for subsequent comparison with the Bayesian Network.

#### 5.4 Comparison of Logistic Regression Baselines

The performance of the unweighted and class-weighted logistic regression models was compared directly to summarise the effect of accounting for class imbalance.

In [16]:
baseline_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Weighted Logistic Regression"
    ],
    "Accuracy": [
        accuracy,
        weighted_accuracy
    ],
    "Precision": [
        precision,
        weighted_precision
    ],
    "Recall": [
        recall,
        weighted_recall
    ],
    "F1": [
        f1,
        weighted_f1
    ],
    "ROC-AUC": [
        roc_auc,
        weighted_roc_auc
    ],
    "PR-AUC": [
        pr_auc,
        weighted_pr_auc
    ]
})

baseline_comparison.round(4)

,Model,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Logistic Regression,0.9927,0.6391,0.2193,0.3266,0.9502,0.4776
1,Weighted Logistic Regression,0.9644,0.1694,0.8671,0.2834,0.9567,0.3941


### 6. Bayesian Network Modelling

#### 6.1 Preparation of Bayesian Network Dataset

A discrete session-level dataset was used for Bayesian Network modelling. Continuous and count-based behavioural variables were discretised during preprocessing because the Bayesian Network implementation used in this study models discrete variables through conditional probability tables.

The dataset contains behavioural, temporal and conversion variables selected during exploratory analysis and feature engineering. Conversion (`converted`) represents the target outcome. The remaining variables provide candidate explanatory and predictive factors for structure learning.

Before structure learning, the processed dataset was reloaded and checked for missing values, variable states and class distribution to verify that it was suitable for discrete Bayesian Network modelling.

In [17]:
import pandas as pd

bn_data = pd.read_csv("../data/processed/bn_discrete.csv")

print(f"Number of sessions: {len(bn_data):,}")
print(f"Number of variables: {bn_data.shape[1]}")

bn_data.head()

Number of sessions: 1,761,675
Number of variables: 7


,event_count,unique_items,has_addtocart,day_of_week,converted,session_duration,time_of_day
0,2-3,2-3,0,Friday,0,5-15,evening
1,1,1,0,Friday,0,0,evening
2,1,1,0,Tuesday,0,0,evening
3,2-3,2-3,0,Friday,0,15+,evening
4,1,1,0,Wednesday,0,0,evening


In [18]:
# Check dataset before Bayesian Network modelling
print("Missing values:")
print(bn_data.isna().sum())

print("\nUnique states per variable:")
for column in bn_data.columns:
    print(f"{column}: {bn_data[column].nunique()} states")

Missing values:
event_count         0
unique_items        0
has_addtocart       0
day_of_week         0
converted           0
session_duration    0
time_of_day         0
dtype: int64

Unique states per variable:
event_count: 3 states
unique_items: 3 states
has_addtocart: 2 states
day_of_week: 7 states
converted: 2 states
session_duration: 4 states
time_of_day: 4 states


##### 6.1.1 Bayesian Network Train-Test Split

To ensure that subsequent predictive evaluation is performed on genuinely unseen observations, the discretised Bayesian Network dataset was divided using the same training and test indices established for the logistic regression models. This preserves an identical evaluation sample across modelling approaches while preventing observations in the held-out test set from contributing to Bayesian Network structure or parameter learning.

In [19]:
# Create Bayesian Network training and test sets
# using the same indices as the logistic regression split

bn_train = bn_data.loc[X_train.index].copy()
bn_test = bn_data.loc[X_test.index].copy()

print(f"BN training observations: {len(bn_train):,}")
print(f"BN test observations: {len(bn_test):,}")

print("\nTraining conversion rate:")
print(f"{bn_train['converted'].mean() * 100:.2f}%")

print("\nTest conversion rate:")
print(f"{bn_test['converted'].mean() * 100:.2f}%")

BN training observations: 1,409,340
BN test observations: 352,335

Training conversion rate:
0.81%

Test conversion rate:
0.81%


#### 6.2 Bayesian Network Structure Learning

The structure of the Bayesian Network was learned from the discretised **training data only**, ensuring that the held-out test observations did not influence either network structure selection or subsequent parameter estimation.

Hill Climb Search was used to identify a directed acyclic graph (DAG) representing statistical dependencies between the session characteristics and conversion outcome. The Bayesian Information Criterion (BIC) was initially used as the scoring criterion. BIC balances model fit against model complexity, penalising the inclusion of additional parameters that do not provide sufficient improvement in fit.

The resulting network therefore represents the dependency structure identified from the training observations, while the held-out test set remains unseen for later predictive evaluation.

In [20]:
import pgmpy

print("pgmpy version:", pgmpy.__version__)

pgmpy version: 0.1.23


In [21]:
from pgmpy.estimators import HillClimbSearch, BicScore

In [22]:
# Initialise BIC scoring method using training data only
bic = BicScore(bn_train)

# Learn Bayesian Network structure from training data only
hc_bic = HillClimbSearch(bn_train)

hc_bic_model = hc_bic.estimate(
    scoring_method=bic
)

print("BIC learned edges:")
for edge in hc_bic_model.edges():
    print(edge)

print(f"\nNumber of edges: {len(hc_bic_model.edges())}")

  0%|          | 0/1000000 [00:00<?, ?it/s]

BIC learned edges:
('event_count', 'session_duration')
('event_count', 'has_addtocart')
('event_count', 'converted')
('unique_items', 'event_count')
('unique_items', 'has_addtocart')
('unique_items', 'session_duration')
('unique_items', 'converted')
('has_addtocart', 'converted')
('has_addtocart', 'session_duration')
('day_of_week', 'time_of_day')
('session_duration', 'converted')
('time_of_day', 'unique_items')

Number of edges: 12


In [23]:
print("Nodes:")
print(list(hc_bic_model.nodes()))

print("\nEdges:")
for edge in hc_bic_model.edges():
    print(edge)

Nodes:
['event_count', 'unique_items', 'has_addtocart', 'day_of_week', 'converted', 'session_duration', 'time_of_day']

Edges:
('event_count', 'session_duration')
('event_count', 'has_addtocart')
('event_count', 'converted')
('unique_items', 'event_count')
('unique_items', 'has_addtocart')
('unique_items', 'session_duration')
('unique_items', 'converted')
('has_addtocart', 'converted')
('has_addtocart', 'session_duration')
('day_of_week', 'time_of_day')
('session_duration', 'converted')
('time_of_day', 'unique_items')


#### 6.3 Alternative Structure Learning Using K2 Score

To assess the robustness of the learned network structure to the choice of scoring criterion, a second Bayesian Network structure was estimated using Hill Climb Search with the K2 score. Keeping the search algorithm constant allows differences between the resulting structures to be attributed primarily to the scoring criterion. The learned structures are subsequently compared in terms of their edges and relationships involving the conversion outcome.

In [24]:
from pgmpy.estimators import K2Score

# Initialise K2 scoring method using training data only
k2 = K2Score(bn_train)

# Learn alternative network structure from training data only
hc_k2 = HillClimbSearch(bn_train)

hc_k2_model = hc_k2.estimate(
    scoring_method=k2
)

print("K2 learned edges:")
for edge in hc_k2_model.edges():
    print(edge)

print(f"\nNumber of edges: {len(hc_k2_model.edges())}")

  0%|          | 0/1000000 [00:00<?, ?it/s]

K2 learned edges:
('event_count', 'unique_items')
('event_count', 'has_addtocart')
('event_count', 'converted')
('unique_items', 'has_addtocart')
('unique_items', 'converted')
('converted', 'time_of_day')
('converted', 'day_of_week')
('converted', 'has_addtocart')
('session_duration', 'event_count')
('session_duration', 'converted')
('session_duration', 'unique_items')
('session_duration', 'has_addtocart')
('session_duration', 'time_of_day')
('time_of_day', 'day_of_week')

Number of edges: 14


#### 6.4 Comparison of Learned Structures

The structures learned from the training data using BIC and K2 scoring were compared to assess the sensitivity of the Bayesian Network to the choice of scoring criterion. Because the same Hill Climb Search algorithm and training observations were used for both models, differences between the resulting structures primarily reflect the alternative scoring criteria.

Particular attention was given to relationships involving the conversion outcome, as these are most relevant to the study objectives. Agreement between the learned networks suggests comparatively stable dependencies, while differences indicate structural uncertainty that should be considered when interpreting individual edge orientations.

In [25]:
# Convert learned edges to sets
bic_edges = set(hc_bic_model.edges())
k2_edges = set(hc_k2_model.edges())

# Compare exact directed edges
common_edges = bic_edges & k2_edges
bic_only = bic_edges - k2_edges
k2_only = k2_edges - bic_edges

print(f"BIC edges: {len(bic_edges)}")
print(f"K2 edges: {len(k2_edges)}")
print(f"Common directed edges: {len(common_edges)}")

print("\nCommon edges:")
for edge in sorted(common_edges):
    print(edge)

print("\nBIC-only edges:")
for edge in sorted(bic_only):
    print(edge)

print("\nK2-only edges:")
for edge in sorted(k2_only):
    print(edge)

BIC edges: 12
K2 edges: 14
Common directed edges: 5

Common edges:
('event_count', 'converted')
('event_count', 'has_addtocart')
('session_duration', 'converted')
('unique_items', 'converted')
('unique_items', 'has_addtocart')

BIC-only edges:
('day_of_week', 'time_of_day')
('event_count', 'session_duration')
('has_addtocart', 'converted')
('has_addtocart', 'session_duration')
('time_of_day', 'unique_items')
('unique_items', 'event_count')
('unique_items', 'session_duration')

K2-only edges:
('converted', 'day_of_week')
('converted', 'has_addtocart')
('converted', 'time_of_day')
('event_count', 'unique_items')
('session_duration', 'event_count')
('session_duration', 'has_addtocart')
('session_duration', 'time_of_day')
('session_duration', 'unique_items')
('time_of_day', 'day_of_week')


In [26]:
def undirected_edges(model):
    return {
        frozenset(edge)
        for edge in model.edges()
    }

bic_connections = undirected_edges(hc_bic_model)
k2_connections = undirected_edges(hc_k2_model)

common_connections = bic_connections & k2_connections

print(
    f"Common connections regardless of direction: "
    f"{len(common_connections)}"
)

print("\nCommon connections:")
for connection in common_connections:
    print(tuple(connection))

Common connections regardless of direction: 11

Common connections:
('has_addtocart', 'session_duration')
('unique_items', 'has_addtocart')
('unique_items', 'converted')
('event_count', 'has_addtocart')
('event_count', 'converted')
('converted', 'session_duration')
('event_count', 'session_duration')
('unique_items', 'session_duration')
('converted', 'has_addtocart')
('time_of_day', 'day_of_week')
('event_count', 'unique_items')


##### Comparison of BIC and K2 Structures

The BIC and K2 scoring criteria produced networks containing 12 and 14 directed edges, respectively. OSix directed edges were identical between the two structures. However, when edge direction was ignored, 11 connections were shared between the networks. This indicates that the two scoring criteria identified a broadly similar underlying dependency structure, while differing substantially in the orientation assigned to several relationships.

Importantly, several relationships involving the conversion outcome were stable across both structures. Both networks contained the directed relationships `event_count → converted`, `unique_items → converted`, and `session_duration → converted`. A connection between `has_addtocart` and `converted` was also identified by both networks, although its direction differed: BIC selected `has_addtocart → converted`, whereas K2 selected `converted → has_addtocart`.

Similar orientation differences occurred elsewhere in the network. For example, BIC selected `event_count → session_duration`, while K2 selected `session_duration → event_count`. The relationship between `unique_items` and `event_count` was likewise present in both structures but with opposite orientations. These results demonstrate that the existence of several dependencies is relatively robust to the choice of scoring criterion, while their inferred direction is more sensitive.

The BIC structure was retained for subsequent parameter estimation and probabilistic inference. BIC explicitly penalises model complexity and produced the more parsimonious network, with 12 edges compared with 14 under K2. Given the observational nature of the data and the sensitivity of some edge orientations to the scoring criterion, the learned directions are interpreted as statistical dependencies rather than evidence of causal relationships.

#### 6.5 Bayesian Network Parameter Estimation

Following structure learning, the parameters of the selected BIC network were estimated using the training observations only. This ensures that the held-out test data remained unseen during both structure and parameter learning and could subsequently be used for an unbiased evaluation of predictive performance.

For each node, a conditional probability distribution was estimated conditional on the states of its parent nodes. These conditional probability distributions form the Conditional Probability Tables (CPTs) of the Bayesian Network.

Maximum Likelihood Estimation (MLE) was used so that each conditional probability was derived directly from the observed relative frequencies within the training data. For a node \(X\) with parent configuration \(Pa(X)\), the estimated conditional probability is:

$$
P(X=x \mid Pa(X)=pa)
=
\frac{N(X=x,\ Pa(X)=pa)}
{N(Pa(X)=pa)}
$$

where \(N(X=x,\ Pa(X)=pa)\) represents the number of training observations containing the specified child and parent states, and \(N(Pa(X)=pa)\) represents the total number of training observations with that parent configuration.

In [27]:
from pgmpy.models import BayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator

# Construct Bayesian Network using BIC-learned structure
bn_model = BayesianNetwork(hc_bic_model.edges())

# Estimate CPT parameters using training data only
bn_model.fit(
    bn_train,
    estimator=MaximumLikelihoodEstimator
)

print("Model fitted successfully.")
print(f"Number of nodes: {len(bn_model.nodes())}")
print(f"Number of edges: {len(bn_model.edges())}")
print(f"Number of CPDs: {len(bn_model.get_cpds())}")

print("\nModel valid:", bn_model.check_model())

Model fitted successfully.
Number of nodes: 7
Number of edges: 12
Number of CPDs: 7

Model valid: True


#### 6.6 Conditional Probability Distribution for Conversion

The Conditional Probability Table (CPT) for the conversion node was examined to understand how the estimated probability of conversion varies across different configurations of its parent variables. Under the BIC-learned structure, the parents of `converted` determine the conditional probabilities represented by:

$$
P(\text{converted} \mid Pa(\text{converted}))
$$

Examining these probabilities provides an interpretable representation of how conversion likelihood differs across behavioural and temporal session characteristics within the learned Bayesian Network.

In [28]:
# Identify parents of the conversion node
conversion_parents = list(
    bn_model.get_parents("converted")
)

print("Parents of converted:")
print(conversion_parents)

Parents of converted:
['event_count', 'unique_items', 'has_addtocart', 'session_duration']


In [29]:
# Retrieve the Conditional Probability Table for conversion
converted_cpd = bn_model.get_cpds("converted")

print(converted_cpd)

+------------------+-----+------------------------+
| event_count      | ... | event_count(4+)        |
+------------------+-----+------------------------+
| has_addtocart    | ... | has_addtocart(1)       |
+------------------+-----+------------------------+
| session_duration | ... | session_duration(5-15) |
+------------------+-----+------------------------+
| unique_items     | ... | unique_items(4+)       |
+------------------+-----+------------------------+
| converted(0)     | ... | 0.8611275964391691     |
+------------------+-----+------------------------+
| converted(1)     | ... | 0.13887240356083086    |
+------------------+-----+------------------------+


#### 6.7 Verification of Conditional Probability Estimation

To verify the parameter estimation process, an individual conditional probability from the conversion CPT was reproduced directly from the training data. For a specified combination of parent states, the probability of conversion was calculated as:

$$
P(\text{converted}=1 \mid Pa(\text{converted})=pa)
=
\frac{
N(\text{converted}=1,\ Pa(\text{converted})=pa)
}{
N(Pa(\text{converted})=pa)
}
$$

The manually calculated probability was then compared with the corresponding value stored in the fitted Bayesian Network CPT. Agreement between the two values verifies that the Maximum Likelihood Estimate represents the observed relative frequency for the specified parent configuration within the training data.

In [30]:
# Define parent configuration
configuration = {
    "event_count": "4+",
    "unique_items": "4+",
    "has_addtocart": 1,
    "session_duration": "5-15"
}

# Select TRAINING sessions matching the parent configuration
mask = (
    (bn_train["event_count"] == configuration["event_count"]) &
    (bn_train["unique_items"] == configuration["unique_items"]) &
    (bn_train["has_addtocart"] == configuration["has_addtocart"]) &
    (bn_train["session_duration"] == configuration["session_duration"])
)

matching_sessions = bn_train.loc[mask]

# Calculate probability manually
total_matching = len(matching_sessions)
converted_matching = matching_sessions["converted"].sum()

manual_probability = converted_matching / total_matching

print(f"Matching training sessions: {total_matching:,}")
print(f"Converted training sessions: {converted_matching:,}")
print(f"Manual P(converted=1): {manual_probability:.6f}")

Matching training sessions: 1,685
Converted training sessions: 234
Manual P(converted=1): 0.138872


In [31]:
# Retrieve the same probability from the fitted CPT
cpt_probability = converted_cpd.get_value(
    converted=1,
    event_count="4+",
    unique_items="4+",
    has_addtocart=1,
    session_duration="5-15"
)

print(f"CPT P(converted=1):    {cpt_probability:.6f}")
print(f"Manual probability:    {manual_probability:.6f}")
print(
    f"Difference:            "
    f"{abs(cpt_probability - manual_probability):.10f}"
)

CPT P(converted=1):    0.138872
Manual probability:    0.138872
Difference:            0.0000000000


The selected parent configuration occurred in 1,685 training sessions, of which 234 resulted in conversion. The manually calculated conditional conversion probability was therefore approximately 0.1389 (13.89%). This exactly matched the probability stored in the fitted CPT, with a difference of 0.0000. This verifies that the Maximum Likelihood Estimate for this CPT entry corresponds to the observed relative frequency of conversion for the specified combination of parent states within the training data.

#### 6.8 Probabilistic Inference for Conversion

Following parameter estimation, probabilistic inference was performed using the fitted Bayesian Network. This allows the probability of conversion to be estimated both without additional evidence and conditional on observed session characteristics.

The unconditional probability of conversion provides a baseline:

$$
P(\text{converted}=1)
$$

Conditional probabilities can then be calculated after introducing evidence about other variables in the network. For example, the effect of add-to-cart behaviour can be examined by comparing:

$$
P(\text{converted}=1 \mid \text{has_addtocart}=1)
$$

with:

$$
P(\text{converted}=1 \mid \text{has_addtocart}=0)
$$

These inference queries provide an interpretable way to examine how the model's estimated probability of conversion changes when different session characteristics are observed.

##### 6.8.1 Baseline Conversion Probability

The marginal probability of conversion was first calculated without supplying any evidence. This represents the conversion probability estimated by the Bayesian Network across all sessions and provides a reference point for subsequent conditional inference queries.

In [32]:
from pgmpy.inference import VariableElimination

# Initialise exact inference
inference = VariableElimination(bn_model)

# Query baseline probability of conversion
baseline_conversion = inference.query(
    variables=["converted"]
)

print("Baseline probability of conversion:")
print(baseline_conversion)

baseline_prob = baseline_conversion.get_value(converted=1)

print(
    f"\nP(converted=1): "
    f"{baseline_prob:.6f} "
    f"({baseline_prob * 100:.2f}%)"
)

Baseline probability of conversion:
+--------------+------------------+
| converted    |   phi(converted) |
+==============+==================+
| converted(0) |           0.9919 |
+--------------+------------------+
| converted(1) |           0.0081 |
+--------------+------------------+

P(converted=1): 0.008116 (0.81%)


The Bayesian Network estimated an unconditional conversion probability of approximately 0.81%. This is consistent with the conversion rate observed in the training data, indicating that the fitted network reproduces the marginal distribution of the conversion outcome used during parameter estimation.

This probability provides a baseline against which subsequent conditional probabilities can be compared. Changes from the 0.81% baseline after introducing evidence indicate how the model updates the estimated likelihood of conversion given particular observed session characteristics.

##### 6.8.2 Conversion Probability Conditional on Add-to-Cart Behaviour

Add-to-cart behaviour was identified as a direct parent of the conversion node in the BIC-learned network. Conditional inference was therefore used to examine how the estimated probability of conversion changes depending on whether an add-to-cart event occurred during the session.

The following probabilities were estimated:

$$
P(\text{converted}=1 \mid \text{has_addtocart}=0)
$$

and

$$
P(\text{converted}=1 \mid \text{has_addtocart}=1)
$$

These probabilities were compared with the unconditional conversion probability to quantify how the model updates conversion likelihood when add-to-cart behaviour is observed.

In [33]:
# Conversion probability when no add-to-cart event occurred
no_cart_query = inference.query(
    variables=["converted"],
    evidence={"has_addtocart": 0}
)

# Conversion probability when an add-to-cart event occurred
cart_query = inference.query(
    variables=["converted"],
    evidence={"has_addtocart": 1}
)

no_cart_prob = no_cart_query.get_value(converted=1)
cart_prob = cart_query.get_value(converted=1)

print(f"Baseline conversion probability: {baseline_prob:.6f} ({baseline_prob * 100:.2f}%)")
print(f"P(converted=1 | has_addtocart=0): {no_cart_prob:.6f} ({no_cart_prob * 100:.2f}%)")
print(f"P(converted=1 | has_addtocart=1): {cart_prob:.6f} ({cart_prob * 100:.2f}%)")

print(
    f"\nProbability ratio (add-to-cart vs no add-to-cart): "
    f"{cart_prob / no_cart_prob:.2f}x"
)

Baseline conversion probability: 0.008116 (0.81%)
P(converted=1 | has_addtocart=0): 0.001359 (0.14%)
P(converted=1 | has_addtocart=1): 0.272720 (27.27%)

Probability ratio (add-to-cart vs no add-to-cart): 200.63x


The inferred probability of conversion differed substantially according to add-to-cart behaviour. For sessions without an add-to-cart event, the estimated probability of conversion was approximately 0.14%, below the unconditional baseline of 0.81%. In contrast, when an add-to-cart event was observed, the estimated conversion probability increased to approximately 27.27%.

The estimated probability of conversion was therefore approximately 201 times higher for sessions containing an add-to-cart event than for sessions without one. This indicates that add-to-cart behaviour is strongly associated with conversion within the fitted Bayesian Network and represents an important behavioural signal for distinguishing higher-conversion sessions.

These results describe conditional associations captured by the learned network and should not be interpreted as evidence that add-to-cart behaviour itself causes conversion.

##### 6.8.3 Conversion Probability by Session Event Count

The number of events within a session was also identified as a direct parent of the conversion node. Probabilistic inference was therefore used to estimate conversion probability across the three discretised event-count states.

This allows the relationship identified during exploratory analysis to be examined within the fitted Bayesian Network by estimating:

$$
P(\text{converted}=1 \mid \text{event_count}=e)
$$

for each event-count category \(e\).

In [34]:
# Estimate conversion probability for each event-count state
event_states = ["1", "2-3", "4+"]

event_results = []

for state in event_states:
    query = inference.query(
        variables=["converted"],
        evidence={"event_count": state}
    )
    
    probability = query.get_value(converted=1)
    
    event_results.append({
        "event_count": state,
        "conversion_probability": probability,
        "conversion_percent": probability * 100
    })

event_inference = pd.DataFrame(event_results)

print("Conversion probability by event count:")
display(event_inference.round({
    "conversion_probability": 6,
    "conversion_percent": 2
}))

Conversion probability by event count:


,event_count,conversion_probability,conversion_percent
0,1,0.000378,0.04
1,2-3,0.016336,1.63
2,4+,0.093738,9.37


The inferred probability of conversion increased substantially with session event count. Sessions containing only one event had an estimated conversion probability of approximately 0.04%, compared with 1.63% for sessions containing two to three events and 9.37% for sessions containing four or more events.

The probability for the highest event-count category was therefore substantially above the unconditional conversion probability of 0.81%. This pattern is consistent with the exploratory analysis, which also showed increasing conversion rates as the number of events within a session increased.

The results suggest that greater session activity is strongly associated with higher conversion likelihood within the fitted Bayesian Network. However, these conditional probabilities represent associations captured by the model rather than evidence of a causal effect of additional events on conversion.

##### 6.8.4 Conversion Probability Under Combined Evidence

A key advantage of Bayesian Networks is their ability to update probabilities when multiple pieces of evidence are observed simultaneously. To demonstrate this, conversion probabilities were estimated for contrasting session profiles combining event-count and add-to-cart information.

A lower-engagement profile was defined as a session containing a single event and no add-to-cart behaviour, while a higher-engagement profile contained four or more events and an observed add-to-cart event.

The corresponding conditional probabilities were estimated as:

$$
P(\text{converted}=1 \mid \text{event_count}=1,\ \text{has_addtocart}=0)
$$

and

$$
P(\text{converted}=1 \mid \text{event_count}=4+,\ \text{has_addtocart}=1)
$$

This comparison illustrates how multiple observed session characteristics can be incorporated simultaneously when estimating conversion probability.

In [35]:
# Lower-engagement session profile
low_engagement_query = inference.query(
    variables=["converted"],
    evidence={
        "event_count": "1",
        "has_addtocart": 0
    }
)

# Higher-engagement session profile
high_engagement_query = inference.query(
    variables=["converted"],
    evidence={
        "event_count": "4+",
        "has_addtocart": 1
    }
)

low_engagement_prob = low_engagement_query.get_value(converted=1)
high_engagement_prob = high_engagement_query.get_value(converted=1)

print(
    f"Low-engagement profile: "
    f"{low_engagement_prob:.6f} "
    f"({low_engagement_prob * 100:.2f}%)"
)

print(
    f"High-engagement profile: "
    f"{high_engagement_prob:.6f} "
    f"({high_engagement_prob * 100:.2f}%)"
)

print(
    f"\nProbability ratio: "
    f"{high_engagement_prob / low_engagement_prob:.2f}x"
)

Low-engagement profile: 0.000379 (0.04%)
High-engagement profile: 0.392230 (39.22%)

Probability ratio: 1034.93x


The combined-evidence queries produced a substantial difference in estimated conversion probability between the two session profiles. A session containing a single event and no add-to-cart behaviour had an estimated conversion probability of approximately 0.04%. In contrast, a session containing four or more events together with an add-to-cart event had an estimated conversion probability of approximately 39.22%.

The estimated conversion probability for the higher-engagement profile was approximately 1,035 times that of the lower-engagement profile. More importantly, the analysis demonstrates how the Bayesian Network can combine multiple observed session characteristics when updating the probability of conversion, rather than considering each characteristic independently.

The results indicate that sessions exhibiting both greater activity and add-to-cart behaviour are associated with substantially higher conversion likelihood within the fitted network. As the network was learned from observational data, these differences represent probabilistic associations and should not be interpreted as causal effects.

#### 6.9 Bayesian Network Prediction and Model Evaluation

Following parameter estimation and probabilistic inference, the predictive performance of the Bayesian Network was evaluated on held-out test data. This allows the network to be assessed not only as an interpretable probabilistic model, but also as a classifier for session-level conversion.

To maintain consistency with the baseline modelling approach, the Bayesian Network was evaluated using the same 20% test set established previously. Predictions were generated using the observed predictor variables for each test session, while `converted` was treated as the target variable.

Given the substantial class imbalance in the conversion outcome, performance was assessed using accuracy, precision, recall, F1-score, ROC-AUC and Precision-Recall AUC (PR-AUC). Particular emphasis was placed on recall, F1-score and PR-AUC, as accuracy alone can be misleading when the positive class is rare.

##### 6.9.1 Prediction on Held-Out Test Data

The Bayesian Network requires discrete input states corresponding to those used during structure and parameter learning. Therefore, the observations belonging to the held-out test set were retrieved from the discretised dataset while preserving the same train-test partition used for the logistic regression baseline.

For prediction, the conversion outcome was removed from the evidence supplied to the network. The remaining variables were used as observed evidence from which the most probable conversion state could be predicted.

In [36]:
# Construct Bayesian Network test set using the same test indices
bn_test = bn_data.loc[X_test.index].copy()

print(f"Bayesian Network test observations: {len(bn_test):,}")
print(f"Expected test observations: {len(X_test):,}")

print("\nConversion distribution:")
print(bn_test["converted"].value_counts())

print("\nConversion rate:")
print(f"{bn_test['converted'].mean() * 100:.2f}%")

Bayesian Network test observations: 352,335
Expected test observations: 352,335

Conversion distribution:
converted
0    349476
1      2859
Name: count, dtype: int64

Conversion rate:
0.81%


##### 6.9.2 Bayesian Network Predictions

For each held-out session, the Bayesian Network was used to estimate the posterior probability of conversion conditional on the observed session characteristics. As the predictor variables were discretised, many test observations shared identical evidence configurations. To avoid performing repeated inference for identical combinations, posterior conversion probabilities were calculated once for each unique evidence configuration and subsequently mapped back to the corresponding test observations.

The predicted probability for each session is represented by:

$$
P(\text{converted}=1 \mid E)
$$

where $E$ represents the observed evidence consisting of the discretised session characteristics.

A classification threshold of 0.5 was initially applied to convert the posterior probabilities into binary conversion predictions, consistent with the standard threshold used for the unweighted logistic regression baseline.

In [37]:
# Ensure inference uses the corrected training-only Bayesian Network
inference = VariableElimination(bn_model)

In [38]:
evidence_columns = [
    "event_count",
    "unique_items",
    "has_addtocart",
    "day_of_week",
    "session_duration",
    "time_of_day"
]

unique_evidence = (
    bn_test[evidence_columns]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"Unique evidence configurations: {len(unique_evidence):,}")

Unique evidence configurations: 904


In [39]:
# Estimate conversion probability for each unique evidence configuration
probabilities = []

for _, row in unique_evidence.iterrows():
    evidence = {
        column: row[column]
        for column in evidence_columns
    }

    query = inference.query(
        variables=["converted"],
        evidence=evidence,
        show_progress=False
    )

    probability = query.get_value(converted=1)
    probabilities.append(probability)

# Add probabilities to unique evidence table
unique_evidence["bn_conversion_probability"] = probabilities

print("Inference completed successfully.")
print(f"Probabilities calculated: {len(probabilities):,}")

print("\nColumns:")
print(unique_evidence.columns.tolist())

display(unique_evidence.head(10))

Inference completed successfully.
Probabilities calculated: 904

Columns:
['event_count', 'unique_items', 'has_addtocart', 'day_of_week', 'session_duration', 'time_of_day', 'bn_conversion_probability']


,event_count,unique_items,has_addtocart,day_of_week,session_duration,time_of_day,bn_conversion_probability
0,1,1,0,Friday,0,morning,0.000379
1,2-3,2-3,0,Sunday,0-5,night,0.000960
2,4+,4+,0,Thursday,5-15,afternoon,0.003320
3,1,1,0,Tuesday,0,evening,0.000379
4,4+,4+,0,Sunday,5-15,afternoon,0.003320
5,2-3,1,0,Monday,0-5,evening,0.007876
6,1,1,0,Sunday,0,afternoon,0.000379
7,1,1,0,Tuesday,0,night,0.000379
8,4+,4+,1,Monday,0-5,morning,0.044860
9,1,1,0,Monday,0,night,0.000379


In [40]:
# Map probabilities back to all held-out test observations
bn_predictions = bn_test.merge(
    unique_evidence[
        evidence_columns + ["bn_conversion_probability"]
    ],
    on=evidence_columns,
    how="left"
)

print(f"Test observations: {len(bn_predictions):,}")
print(
    "Missing predicted probabilities:",
    bn_predictions["bn_conversion_probability"].isna().sum()
)

display(
    bn_predictions[
        ["converted", "bn_conversion_probability"]
    ].head(10)
)

Test observations: 352,335
Missing predicted probabilities: 0


,converted,bn_conversion_probability
0,0,0.000379
1,0,0.000960
2,0,0.003320
3,0,0.000379
4,0,0.003320
5,0,0.007876
6,0,0.000379
7,0,0.000379
8,0,0.044860
9,0,0.000379


##### 6.9.3 Predictive Performance

The posterior conversion probabilities were converted into binary predictions using an initial classification threshold of 0.5. Predictive performance was then evaluated using the same metrics applied to the logistic regression baselines: accuracy, precision, recall, F1-score, ROC-AUC and PR-AUC.

ROC-AUC and PR-AUC were calculated directly from the posterior probabilities rather than the binary classifications. This provides an assessment of the model's ability to rank converting sessions across possible classification thresholds. PR-AUC is particularly informative in this context because conversion represents a small minority of the observations.

In [41]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

## True outcomes and predicted probabilities
bn_y_true = bn_predictions["converted"]
bn_y_prob = bn_predictions["bn_conversion_probability"]

# Binary predictions using 0.5 threshold
bn_y_pred = (bn_y_prob >= 0.5).astype(int)

# Evaluation metrics
bn_accuracy = accuracy_score(bn_y_true, bn_y_pred)
bn_precision = precision_score(
    bn_y_true, bn_y_pred, zero_division=0
)
bn_recall = recall_score(
    bn_y_true, bn_y_pred, zero_division=0
)
bn_f1 = f1_score(
    bn_y_true, bn_y_pred, zero_division=0
)
bn_roc_auc = roc_auc_score(bn_y_true, bn_y_prob)
bn_pr_auc = average_precision_score(bn_y_true, bn_y_prob)

print(f"Accuracy:  {bn_accuracy:.4f}")
print(f"Precision: {bn_precision:.4f}")
print(f"Recall:    {bn_recall:.4f}")
print(f"F1-score:  {bn_f1:.4f}")
print(f"ROC-AUC:   {bn_roc_auc:.4f}")
print(f"PR-AUC:    {bn_pr_auc:.4f}")

Accuracy:  0.9924
Precision: 0.5671
Recall:    0.2424
F1-score:  0.3396
ROC-AUC:   0.9652
PR-AUC:    0.4430


In [56]:
# Baseline conversion rate in the Bayesian Network test set
baseline_conversion = bn_y_true.mean()

print(f"Baseline conversion probability: {baseline_conversion:.4f}")
print(f"Baseline conversion rate: {baseline_conversion * 100:.2f}%")

Baseline conversion probability: 0.0081
Baseline conversion rate: 0.81%


In [42]:
bn_cm = confusion_matrix(bn_y_true, bn_y_pred)

print("Confusion matrix:")
print(bn_cm)

print("\nClassification report:")
print(
    classification_report(
        bn_y_true,
        bn_y_pred,
        digits=4,
        zero_division=0
    )
)

Confusion matrix:
[[348947    529]
 [  2166    693]]

Classification report:
              precision    recall  f1-score   support

           0     0.9938    0.9985    0.9962    349476
           1     0.5671    0.2424    0.3396      2859

    accuracy                         0.9924    352335
   macro avg     0.7805    0.6204    0.6679    352335
weighted avg     0.9904    0.9924    0.9908    352335



At the default classification threshold of 0.5, the Bayesian Network achieved an accuracy of 99.24%, precision of 56.71%, recall of 24.24%, and an F1-score of 0.3396. Of the 2,859 converting sessions in the held-out test set, the model correctly identified 693, while 2,166 were classified as non-converting. The model additionally produced 529 false-positive conversion predictions.

Although overall accuracy was high, this metric should be interpreted cautiously because only 0.81% of test observations represented conversions. The probability-based metrics therefore provide a more informative assessment of discrimination under this substantial class imbalance. The Bayesian Network achieved a ROC-AUC of 0.9652 and a PR-AUC of 0.4420, indicating strong ranking performance despite the rarity of the positive class.

The results demonstrate that the Bayesian Network provides useful predictive discrimination in addition to its probabilistic and explanatory capabilities. However, recall at the default 0.5 threshold was relatively low, with the model identifying approximately one quarter of converting sessions. This highlights the distinction between probability estimation and binary classification for a rare outcome and suggests that classification performance is sensitive to the choice of decision threshold.

##### 6.9.4 Comparison with Logistic Regression Baselines

The predictive performance of the Bayesian Network was compared with both logistic regression baselines developed previously. This comparison assesses whether the probabilistic and interpretable Bayesian Network approach remains competitive with conventional discriminative classification methods.

In [43]:
# Compare Bayesian Network with logistic regression baselines
model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Weighted Logistic Regression",
        "Bayesian Network"
    ],
    "Accuracy": [
        accuracy,
        weighted_accuracy,
        bn_accuracy
    ],
    "Precision": [
        precision,
        weighted_precision,
        bn_precision
    ],
    "Recall": [
        recall,
        weighted_recall,
        bn_recall
    ],
    "F1": [
        f1,
        weighted_f1,
        bn_f1
    ],
    "ROC-AUC": [
        roc_auc,
        weighted_roc_auc,
        bn_roc_auc
    ],
    "PR-AUC": [
        pr_auc,
        weighted_pr_auc,
        bn_pr_auc
    ]
})

model_comparison.round(4)

,Model,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Logistic Regression,0.9927,0.6391,0.2193,0.3266,0.9502,0.4776
1,Weighted Logistic Regression,0.9644,0.1694,0.8671,0.2834,0.9567,0.3941
2,Bayesian Network,0.9924,0.5671,0.2424,0.3396,0.9652,0.4430


The comparison demonstrates that the Bayesian Network achieved competitive predictive performance relative to the logistic regression baselines. Its accuracy of 99.24% was comparable to the unweighted logistic regression model (99.27%) and higher than the weighted model (96.44%). However, given the substantial class imbalance, accuracy alone provides limited information about performance on the minority conversion class.

The unweighted logistic regression achieved the highest precision (63.91%) and PR-AUC (0.4776), but its recall was relatively low at 21.93%. The Bayesian Network achieved slightly higher recall (24.24%) while maintaining a precision of 56.71%, resulting in the highest F1-score (0.3396) of the three models, compared with 0.3266 for unweighted logistic regression and 0.2834 for weighted logistic regression.

The Bayesian Network also achieved the highest ROC-AUC at 0.9652, compared with 0.9502 and 0.9567 for the unweighted and weighted logistic regression models respectively. Its PR-AUC of 0.4420 was lower than the 0.4776 achieved by unweighted logistic regression but higher than the 0.3941 achieved by weighted logistic regression. This indicates that although the Bayesian Network provided the strongest overall ranking performance according to ROC-AUC, the unweighted logistic regression performed better according to precision-recall performance, which is particularly informative given the rarity of conversion.

Class weighting produced a substantially different operating point for logistic regression. The weighted model achieved by far the highest recall (86.71%) but at the cost of considerably lower precision (16.94%) and the lowest F1-score (0.2834). This illustrates the trade-off between identifying a larger proportion of converting sessions and generating additional false-positive predictions.

Overall, no single model dominated across all evaluation measures. The Bayesian Network achieved the highest ROC-AUC and F1-score, the unweighted logistic regression achieved the highest precision and PR-AUC, and the weighted logistic regression achieved the highest recall. The Bayesian Network therefore remained competitive as a predictive model while additionally supporting capabilities not provided directly by the logistic regression baselines, including graphical representation of conditional dependencies and probabilistic inference under different combinations of observed evidence.

#### 6.10 Network Interpretation and Explainability

A principal motivation for using a Bayesian Network is its ability to provide an interpretable representation of relationships between variables while supporting explicit probabilistic reasoning. Unlike models that provide predictions without directly representing dependencies between variables, the Bayesian Network expresses the joint probability distribution through a directed acyclic graph (DAG) and a set of Conditional Probability Tables (CPTs).

Explainability was therefore examined at two complementary levels. First, the learned network structure was interpreted to identify the variables directly connected to the conversion outcome and the dependencies represented between session characteristics. Second, the fitted CPTs and probabilistic inference queries were used to quantify how estimated conversion probability changes under different observed session characteristics.

As the network structure was learned from observational data, directed edges are interpreted as statistical dependencies represented by the fitted model rather than evidence of causal relationships.

##### 6.10.1 Interpretation of the Learned Network Structure

The local network structure surrounding the conversion node was examined to identify the variables with direct graphical relationships to conversion. Parent nodes provide the variables upon which the conversion CPT is directly conditioned, while child nodes represent variables connected downstream of conversion in the learned DAG.

Examining the Markov blanket additionally identifies the set of variables required to probabilistically isolate the conversion node from the remainder of the network. This provides a structural perspective on the information most directly relevant to conversion within the fitted Bayesian Network.

In [44]:
# Examine the local structure surrounding conversion
conversion_parents = list(bn_model.get_parents("converted"))
conversion_children = list(bn_model.get_children("converted"))
conversion_markov_blanket = list(
    bn_model.get_markov_blanket("converted")
)

print("Parents of converted:")
print(conversion_parents)

print("\nChildren of converted:")
print(conversion_children)

print("\nMarkov blanket of converted:")
print(conversion_markov_blanket)

Parents of converted:
['event_count', 'unique_items', 'has_addtocart', 'session_duration']

Children of converted:
[]

Markov blanket of converted:
['event_count', 'unique_items', 'has_addtocart', 'session_duration']


The learned training-only BIC structure identifies event_count, unique_items, has_addtocart, and session_duration as direct parents of the conversion node. Consequently, the Conditional Probability Table (CPT) for converted is conditioned directly on these four behavioural characteristics. The fitted network therefore represents conversion probability as varying according to combinations of session activity, product exploration, add-to-cart behaviour, and session duration.

No child nodes were identified for converted in the selected BIC structure. Within the learned DAG, conversion therefore appears as a terminal node, with its probability determined by the states of its four parent variables rather than being represented as a parent of other variables.

The Markov blanket of converted consists of event_count, unique_items, has_addtocart, and session_duration. Because converted has no children, its Markov blanket is equivalent to its set of parent nodes. Conditional on these four variables, the conversion node is probabilistically independent of the remaining variables in the fitted network. In particular, day_of_week and time_of_day are not part of the local dependency structure surrounding conversion in the selected BIC model.

This provides a relatively concise structural explanation of conversion within the Bayesian Network. Rather than requiring all modelled session characteristics to explain the outcome directly, the learned structure identifies four variables as locally relevant to the conversion node. As the network was learned from observational training data, these relationships represent statistical dependencies captured by the model and should not be interpreted as causal effects.

##### 6.10.2 Probability-Based Explanation of Conversion Predictions

In addition to the graphical representation of dependencies, the Bayesian Network provides explainability through explicit probability updates. Rather than only assigning a binary conversion prediction, the model can estimate how the probability of conversion changes when different session characteristics are supplied as evidence.

The inference results in Section 6.8 demonstrated this property across both individual and combined evidence. The unconditional probability of conversion provides a baseline against which evidence-conditioned probabilities can be compared. For an observed evidence configuration \(E\), the change in conversion probability can therefore be represented as:

$$
\Delta P = P(\text{converted}=1 \mid E) - P(\text{converted}=1)
$$

A probability ratio can additionally express the relative change:

$$
R(E) =
\frac{P(\text{converted}=1 \mid E)}
{P(\text{converted}=1)}
$$

These quantities provide an interpretable measure of how the model updates its estimated conversion likelihood after observing particular session characteristics.

In [45]:
# Summarise probability updates for interpretable session evidence
explanation_results = pd.DataFrame({
    "Evidence": [
        "No evidence (baseline)",
        "No add-to-cart",
        "Add-to-cart",
        "1 event",
        "2-3 events",
        "4+ events",
        "1 event + no add-to-cart",
        "4+ events + add-to-cart"
    ],
    "Conversion Probability": [
        baseline_prob,
        no_cart_prob,
        cart_prob,
        event_inference.loc[
            event_inference["event_count"] == "1",
            "conversion_probability"
        ].iloc[0],
        event_inference.loc[
            event_inference["event_count"] == "2-3",
            "conversion_probability"
        ].iloc[0],
        event_inference.loc[
            event_inference["event_count"] == "4+",
            "conversion_probability"
        ].iloc[0],
        low_engagement_prob,
        high_engagement_prob
    ]
})

# Compare each probability with the unconditional baseline
explanation_results["Probability (%)"] = (
    explanation_results["Conversion Probability"] * 100
)

explanation_results["Change vs Baseline (pp)"] = (
    explanation_results["Conversion Probability"] - baseline_prob
) * 100

explanation_results["Ratio vs Baseline"] = (
    explanation_results["Conversion Probability"] / baseline_prob
)

display(
    explanation_results.round({
        "Conversion Probability": 6,
        "Probability (%)": 2,
        "Change vs Baseline (pp)": 2,
        "Ratio vs Baseline": 2
    })
)

,Evidence,Conversion Probability,Probability (%),Change vs Baseline (pp),Ratio vs Baseline
0,No evidence (baseline),0.008116,0.81,0.00,1.00
1,No add-to-cart,0.001359,0.14,-0.68,0.17
2,Add-to-cart,0.272720,27.27,26.46,33.60
3,1 event,0.000378,0.04,-0.77,0.05
4,2-3 events,0.016336,1.63,0.82,2.01
5,4+ events,0.093738,9.37,8.56,11.55
6,1 event + no add-to-cart,0.000379,0.04,-0.77,0.05
7,4+ events + add-to-cart,0.392230,39.22,38.41,48.33


The probability updates demonstrate how the Bayesian Network provides interpretable explanations of estimated conversion likelihood. Without any evidence, the estimated probability of conversion was 0.81%. Observing no add-to-cart behaviour reduced this probability to 0.14%, whereas observing an add-to-cart event increased it to 27.27%, corresponding to an increase of 26.46 percentage points and a probability approximately 33.60 times the baseline.

A similar pattern was observed for session activity. Sessions containing a single event had an estimated conversion probability of only 0.04%, while this increased to 1.63% for sessions containing two to three events and 9.37% for sessions containing four or more events. The latter represents a probability approximately 11.55 times the unconditional baseline.

The largest probability update occurred when multiple pieces of evidence were considered simultaneously. A session containing four or more events together with an add-to-cart event produced an estimated conversion probability of 39.22%, an increase of 38.41 percentage points relative to the 0.81% baseline and approximately 48.33 times the baseline probability. In contrast, a session containing only one event and no add-to-cart behaviour had an estimated conversion probability of approximately 0.04%.

These results illustrate an important interpretability property of the Bayesian Network: estimated conversion probabilities can be examined explicitly as evidence is introduced. This provides more than a binary prediction by showing how the model's estimated conversion likelihood varies across observed behavioural profiles. In conjunction with the learned network structure, the results show that event_count and has_addtocart—both direct parents of converted in the selected BIC network—provide particularly informative examples of probability-based explanation. As the network was learned from observational data, these probability updates represent statistical associations captured by the fitted model rather than causal effects.

##### 6.10.3 Explainability Compared with Logistic Regression

The Bayesian Network and logistic regression baselines provide different forms of model interpretability. Logistic regression represents the relationship between predictor variables and conversion through estimated coefficients, which can be transformed into odds ratios to quantify the direction and magnitude of associations while holding the other model variables constant. The Bayesian Network instead represents dependencies graphically and enables conversion probabilities to be queried directly under different combinations of observed evidence.

This distinction is particularly relevant for the present analysis. The learned Bayesian Network identifies `event_count`, `unique_items`, `has_addtocart`, and `session_duration` as direct parents of the conversion node, making the variables upon which its conversion CPT is conditioned explicit. Probabilistic inference can then translate these dependencies into quantities such as:

$$
P(\text{converted}=1 \mid \text{has_addtocart}=1)
$$

or, when multiple observations are available,

$$
P(\text{converted}=1 \mid \text{event_count}=4+,\ 
\text{has_addtocart}=1)
$$

This provides a different explanatory perspective from logistic regression. Rather than interpreting a model parameter, the Bayesian Network allows the estimated probability of the outcome to be examined directly under specified evidence configurations. For example, the fitted network estimated an unconditional conversion probability of 0.81%, increasing to 27.27% when add-to-cart behaviour was observed and to 39.22% when an add-to-cart event was combined with four or more session events. These probability updates provide an intuitive representation of how the fitted model's estimate changes as evidence is introduced.

The predictive comparison in Section 6.9 also shows that this interpretability was not obtained at the expense of predictive discrimination. The Bayesian Network achieved a ROC-AUC of 0.9652, compared with 0.9502 for unweighted logistic regression and 0.9567 for weighted logistic regression. Its positive-class F1-score of 0.3396 was also slightly higher than both logistic regression baselines (0.3266 and 0.2834, respectively). However, no single model dominated across all measures: the unweighted logistic regression achieved higher precision and PR-AUC, while the weighted logistic regression achieved substantially higher recall.

Overall, the Bayesian Network complements conventional predictive evaluation with an explicit probabilistic representation of the relationships between session characteristics and conversion. Its main interpretability advantage in this analysis is therefore not that logistic regression is inherently uninterpretable, but that the network supports graphical inspection of dependencies and intuitive probability-based queries involving multiple pieces of evidence. These relationships remain associational rather than causal because the network structure and parameters were learned from observational session data.

##### 6.10.4 Logistic Regression Coefficient Interpretation

For comparison, the fitted unweighted logistic regression model was also examined through its estimated coefficients. Logistic regression coefficients represent changes in the log-odds of conversion associated with the encoded predictor variables, while exponentiating a coefficient produces the corresponding odds ratio:

$$
OR_j = e^{\beta_j}
$$

where $\beta_j$ is the fitted coefficient for predictor $j$. An odds ratio greater than one indicates higher estimated odds of conversion, while an odds ratio below one indicates lower estimated odds, holding the other model predictors constant. For categorical predictors, odds ratios are interpreted relative to the omitted reference category.

Examining these values provides a coefficient-based form of interpretability that can be contrasted with the graphical and probability-based explanations produced by the Bayesian Network.

In [46]:
# Inspect categorical encoding to identify the reference category
encoder = logistic_model.named_steps["preprocessor"].named_transformers_["categorical"]

print("Day-of-week categories:")
print(encoder.categories_[0])

print("\nDropped reference category:")
print(encoder.categories_[0][encoder.drop_idx_[0]])

Day-of-week categories:
['Friday' 'Monday' 'Saturday' 'Sunday' 'Thursday' 'Tuesday' 'Wednesday']

Dropped reference category:
Friday


In [47]:
# Extract fitted logistic regression classifier
fitted_logistic = logistic_model.named_steps["classifier"]

# Extract transformed feature names from preprocessing pipeline
feature_names = logistic_model.named_steps[
    "preprocessor"
].get_feature_names_out()

# Create coefficient table
lr_coefficients = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": fitted_logistic.coef_[0]
})

# Convert coefficients to odds ratios
lr_coefficients["Odds Ratio"] = np.exp(
    lr_coefficients["Coefficient"]
)

# Sort by absolute coefficient magnitude
lr_coefficients["Absolute Coefficient"] = (
    lr_coefficients["Coefficient"].abs()
)

lr_coefficients = (
    lr_coefficients
    .sort_values(
        "Absolute Coefficient",
        ascending=False
    )
    .drop(columns="Absolute Coefficient")
)

display(lr_coefficients.round(4))

,Feature,Coefficient,Odds Ratio
4,binary__has_addtocart,4.8778,131.3368
6,categorical__day_of_week_Saturday,-0.8039,0.4476
7,categorical__day_of_week_Sunday,-0.7857,0.4558
1,numeric__unique_items,-0.7565,0.4693
8,categorical__day_of_week_Thursday,-0.5561,0.5734
0,numeric__event_count,0.5458,1.7259
5,categorical__day_of_week_Monday,-0.5270,0.5904
9,categorical__day_of_week_Tuesday,-0.4849,0.6158
10,categorical__day_of_week_Wednesday,-0.4473,0.6393
2,numeric__session_duration_minutes,0.3912,1.4788


The logistic regression coefficients provide a complementary form of model interpretation. The strongest positive association was observed for has_addtocart, with a coefficient of 4.8778 and an odds ratio of approximately 131.34. Holding the other predictors constant, sessions containing an add-to-cart event therefore had substantially higher estimated odds of conversion than sessions without an add-to-cart event.

The standardised event_count variable also had a positive coefficient of 0.5458 and an odds ratio of approximately 1.73, indicating that a one-standard-deviation increase in event count was associated with approximately 1.73 times the estimated odds of conversion, holding the remaining predictors constant. Similarly, a one-standard-deviation increase in session_duration_minutes was associated with approximately 1.48 times the estimated odds of conversion. The coefficient for session_hour was comparatively small, with an odds ratio of approximately 1.10.

In contrast, the standardised unique_items variable had a negative coefficient of −0.7565 and an odds ratio of approximately 0.47 after controlling for the other predictors. This should be interpreted as a conditional association within the fitted model rather than evidence that viewing additional unique items itself reduces conversion likelihood, particularly because unique_items is likely to be related to other measures of session activity included in the model.

The day-of-week coefficients were negative relative to Friday, the omitted reference category. Saturday (OR = 0.45) and Sunday (OR = 0.46) showed the largest negative associations, followed by Thursday (OR = 0.57), Monday (OR = 0.59), Tuesday (OR = 0.62), and Wednesday (OR = 0.64). Holding the other predictors constant, these coefficients indicate lower estimated odds of conversion relative to Friday. They represent conditional associations within the fitted model rather than causal effects of individual weekdays.

Together, these results demonstrate that logistic regression provides interpretable information through coefficients and odds ratios, whereas the Bayesian Network provides a complementary form of explanation through its graphical dependency structure and conditional probability queries. For example, logistic regression quantifies the strong association between add-to-cart behaviour and conversion through an odds ratio of approximately 131.34, while the Bayesian Network allows the corresponding conversion probability to be queried directly, increasing from an unconditional 0.81% to 27.27% when add-to-cart behaviour is observed. These quantities should not be compared numerically as equivalent effect measures, since an odds ratio and a conditional probability have different interpretations.

### 7. Decision Threshold Analysis

The predictive models were initially evaluated using the default classification threshold of 0.5. However, given the substantial class imbalance in the conversion outcome, model performance may be sensitive to the choice of decision threshold.

A threshold analysis was therefore conducted to examine the trade-off between precision, recall and F1-score across alternative classification thresholds. This provides a more complete assessment of predictive behaviour than relying on the default threshold alone.

In [48]:
# Evaluate logistic regression performance across alternative thresholds

thresholds = np.arange(0.05, 0.51, 0.05)

threshold_results = []

for threshold in thresholds:
    y_pred_threshold = (y_prob >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_test, y_pred_threshold, zero_division=0
        ),
        "Recall": recall_score(
            y_test, y_pred_threshold, zero_division=0
        ),
        "F1": f1_score(
            y_test, y_pred_threshold, zero_division=0
        )
    })

threshold_results = pd.DataFrame(threshold_results)

threshold_results.round(4)

,Threshold,Precision,Recall,F1
0,0.05,0.2652,0.8255,0.4015
1,0.10,0.2775,0.8206,0.4148
2,0.15,0.3510,0.7975,0.4875
3,0.20,0.4426,0.6971,0.5414
4,0.25,0.4942,0.5488,0.5201
5,0.30,0.5383,0.4428,0.4859
6,0.35,0.5716,0.3645,0.4451
7,0.40,0.5956,0.3008,0.3997
8,0.45,0.6245,0.2606,0.3677
9,0.50,0.6391,0.2193,0.3266


In [49]:
# Identify the threshold with the highest F1-score

best_threshold_row = threshold_results.loc[
    threshold_results["F1"].idxmax()
]

print(f"Best threshold: {best_threshold_row['Threshold']:.2f}")
print(f"Precision: {best_threshold_row['Precision']:.4f}")
print(f"Recall: {best_threshold_row['Recall']:.4f}")
print(f"F1-score: {best_threshold_row['F1']:.4f}")

Best threshold: 0.20
Precision: 0.4426
Recall: 0.6971
F1-score: 0.5414


In [50]:
# Evaluate logistic regression using the selected threshold

best_threshold = best_threshold_row["Threshold"]
y_pred_optimised = (y_prob >= best_threshold).astype(int)

optimised_accuracy = accuracy_score(y_test, y_pred_optimised)
optimised_precision = precision_score(
    y_test, y_pred_optimised, zero_division=0
)
optimised_recall = recall_score(
    y_test, y_pred_optimised, zero_division=0
)
optimised_f1 = f1_score(
    y_test, y_pred_optimised, zero_division=0
)

print(f"Threshold: {best_threshold:.2f}")
print(f"Accuracy: {optimised_accuracy:.4f}")
print(f"Precision: {optimised_precision:.4f}")
print(f"Recall: {optimised_recall:.4f}")
print(f"F1-score: {optimised_f1:.4f}")

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred_optimised))

Threshold: 0.20
Accuracy: 0.9904
Precision: 0.4426
Recall: 0.6971
F1-score: 0.5414

Confusion matrix:
[[346966   2510]
 [   866   1993]]


### Interpretation of Decision Threshold Analysis

The decision threshold analysis demonstrates that predictive performance for the minority conversion class is sensitive to the classification threshold. Using the default threshold of 0.50, the logistic regression model achieved relatively high precision (0.6391) but low recall (0.2193), identifying only 627 of the 2,859 converting sessions.

Among the thresholds evaluated, a threshold of 0.20 produced the highest F1-score of 0.5414. At this threshold, recall increased substantially to 0.6971, with 1,993 converting sessions correctly identified, while precision decreased to 0.4426. The number of false negatives consequently fell from 2,232 at the default threshold to 866, although false positives increased from 354 to 2,510.

These results illustrate the trade-off between precision and recall when predicting a highly imbalanced outcome. Rather than indicating that 0.20 represents a universally optimal decision threshold, it provides the best balance according to F1-score among the thresholds examined. The appropriate operating threshold would ultimately depend on the relative cost of false-positive and false-negative conversion predictions.

## 8. Bayesian Network Decision Threshold Analysis

The Bayesian Network was initially evaluated using a classification threshold of 0.5. Given the substantial class imbalance in the conversion outcome, a threshold sensitivity analysis was also conducted for the Bayesian Network to examine how alternative decision thresholds affect precision, recall and F1-score.

In [51]:
# Evaluate Bayesian Network performance across alternative thresholds

bn_thresholds = np.arange(0.05, 0.51, 0.05)

bn_threshold_results = []

for threshold in bn_thresholds:
    bn_pred_threshold = (bn_y_prob >= threshold).astype(int)

    bn_threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            bn_y_true, bn_pred_threshold, zero_division=0
        ),
        "Recall": recall_score(
            bn_y_true, bn_pred_threshold, zero_division=0
        ),
        "F1": f1_score(
            bn_y_true, bn_pred_threshold, zero_division=0
        )
    })

bn_threshold_results = pd.DataFrame(bn_threshold_results)

bn_threshold_results.round(4)

,Threshold,Precision,Recall,F1
0,0.05,0.3069,0.8265,0.4476
1,0.10,0.3216,0.8199,0.4620
2,0.15,0.3509,0.7786,0.4838
3,0.20,0.4673,0.6639,0.5485
4,0.25,0.4702,0.6614,0.5496
5,0.30,0.4702,0.6614,0.5496
6,0.35,0.4702,0.6614,0.5496
7,0.40,0.4926,0.5838,0.5343
8,0.45,0.5007,0.5310,0.5154
9,0.50,0.5671,0.2424,0.3396


In [52]:
# Identify threshold(s) producing the highest BN F1-score

max_bn_f1 = bn_threshold_results["F1"].max()

best_bn_thresholds = bn_threshold_results[
    np.isclose(bn_threshold_results["F1"], max_bn_f1)
]

print("Threshold(s) with highest F1-score:")
print(best_bn_thresholds.round(4))

# Use the midpoint of the optimal plateau as the representative threshold
best_bn_threshold = 0.30

print(f"\nRepresentative threshold: {best_bn_threshold:.2f}")
print(f"Maximum F1-score: {max_bn_f1:.4f}")

Threshold(s) with highest F1-score:
   Threshold  Precision  Recall      F1
4       0.25     0.4702  0.6614  0.5496
5       0.30     0.4702  0.6614  0.5496
6       0.35     0.4702  0.6614  0.5496

Representative threshold: 0.30
Maximum F1-score: 0.5496


In [53]:
# Evaluate Bayesian Network at representative F1-maximising threshold

bn_y_pred_030 = (bn_y_prob >= best_bn_threshold).astype(int)

bn_accuracy_030 = accuracy_score(bn_y_true, bn_y_pred_030)
bn_precision_030 = precision_score(
    bn_y_true, bn_y_pred_030, zero_division=0
)
bn_recall_030 = recall_score(
    bn_y_true, bn_y_pred_030, zero_division=0
)
bn_f1_030 = f1_score(
    bn_y_true, bn_y_pred_030, zero_division=0
)
bn_cm_030 = confusion_matrix(bn_y_true, bn_y_pred_030)

print(f"Threshold: {best_bn_threshold:.2f}")
print(f"Accuracy: {bn_accuracy_030:.4f}")
print(f"Precision: {bn_precision_030:.4f}")
print(f"Recall: {bn_recall_030:.4f}")
print(f"F1-score: {bn_f1_030:.4f}")

print("\nConfusion matrix:")
print(bn_cm_030)

Threshold: 0.30
Accuracy: 0.9912
Precision: 0.4702
Recall: 0.6614
F1-score: 0.5496

Confusion matrix:
[[347345   2131]
 [   968   1891]]


### Interpretation of Bayesian Network Threshold Analysis

The Bayesian Network was also sensitive to the classification threshold. At the default threshold of 0.50, the model achieved a precision of 0.5671, recall of 0.2424 and F1-score of 0.3396. This indicates that, although predictions classified as conversions were relatively precise, a large proportion of actual converting sessions were not identified.

The highest F1-score of 0.5496 was observed across thresholds from 0.25 to 0.35. A threshold of 0.30 was selected as a representative point within this plateau. At this threshold, recall increased substantially to 0.6614, with 1,891 of the 2,859 converting sessions correctly identified, while precision decreased to 0.4702. False negatives decreased from 2,166 at the default threshold to 968.

As with logistic regression, these results demonstrate the trade-off between precision and recall when predicting a highly imbalanced outcome. The plateau in F1-score between thresholds of 0.25 and 0.35 also indicates that the Bayesian Network's classification performance was relatively stable within this range, rather than depending on a single narrowly defined threshold.

## 9. Threshold-Adjusted Model Comparison

To provide a more informative comparison under class imbalance, the logistic regression and Bayesian Network models were compared at representative thresholds that maximised F1-score within the evaluated threshold range.

In [55]:
# Threshold-adjusted comparison of Logistic Regression and Bayesian Network

# Logistic Regression predictions at selected threshold
lr_threshold = 0.20
lr_y_pred_020 = (y_prob >= lr_threshold).astype(int)

lr_accuracy_020 = accuracy_score(y_test, lr_y_pred_020)
lr_precision_020 = precision_score(y_test, lr_y_pred_020, zero_division=0)
lr_recall_020 = recall_score(y_test, lr_y_pred_020, zero_division=0)
lr_f1_020 = f1_score(y_test, lr_y_pred_020, zero_division=0)

# Create comparison table
threshold_adjusted_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Bayesian Network"
    ],
    "Threshold": [
        lr_threshold,
        best_bn_threshold
    ],
    "Accuracy": [
        lr_accuracy_020,
        bn_accuracy_030
    ],
    "Precision": [
        lr_precision_020,
        bn_precision_030
    ],
    "Recall": [
        lr_recall_020,
        bn_recall_030
    ],
    "F1": [
        lr_f1_020,
        bn_f1_030
    ]
})

threshold_adjusted_comparison.round(4)

,Model,Threshold,Accuracy,Precision,Recall,F1
0,Logistic Regression,0.2,0.9904,0.4426,0.6971,0.5414
1,Bayesian Network,0.3,0.9912,0.4702,0.6614,0.5496
